# Eksik hız verisini GPS'ten türetme

## Problem

Önceki notebook'ta (`1_data_preprocess.ipynb`) gördüğümüz gibi, 70591
koşu antrenmanının sadece ~11500'ünde doğrudan kayıtlı bir `speed`
(hız) verisi var — geri kalan ~60000 antrenmanda bu alan boş. Ama her
antrenmanda GPS konumu (enlem/boylam) ve her ölçümün zaman damgası
kayıtlı. Bu ikisi birlikte hızı hesaplamaya yeter: bir kişi iki GPS
noktası arasında ne kadar mesafe kat etmiş, bunu ne kadar sürede
yapmış — mesafe/zaman = hız.

## Bu notebook'ta yapacaklarımız (özet)

1. **Mesafe hesabı (haversine formülü)** — iki GPS koordinatı arasındaki
   gerçek (küre yüzeyindeki) mesafeyi metre cinsinden hesaplayan
   matematiksel bir formül.
2. **Ham hız türetme** — ardışık noktalar arası mesafeyi, aralarındaki
   zamana bölerek km/h cinsinden hız elde etmek.
3. **Veri kalitesi sorunlarını çözme** — GPS ve zaman damgası verisi
   kusursuz değil (tekrarlanan zaman damgaları, sinyal sıçramaları);
   bunları düzeltmeden ham hız kullanılamayacak kadar gürültülü çıkıyor.
4. **Yumuşatma (Savitzky-Golay filtresi)** — kalan küçük gürültüyü
   bastırıp, gerçek hız değişimini (hızlanma/yavaşlama) koruyarak
   pürüzsüz bir hız eğrisi elde etmek.

Bu notebook'un ilk yarısı, yöntemi TEK bir antrenman üzerinde adım adım
geliştirip doğruluyor; ikinci yarısında aynı yöntem tüm 70591 satıra
uygulanıp sonuç kaydediliyor.

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')

Mounted at /content/drive


In [1]:
import pandas as pd
import numpy as np
from scipy.signal import savgol_filter
from sklearn.metrics import mean_absolute_error, mean_squared_error


In [2]:
PATH = "/home/can/zero-to-ai-architect/runsight/endomondoHR_proper.csv"

df = pd.read_csv(PATH) 

## Veriyi tanı: sütunlar CSV'de neye benziyor

Haversine fonksiyonunu kullanabilmek için önce `latitude`/`longitude`
sütunlarının gerçekte ne tür veri içerdiğini görmemiz lazım.
`df.info()` çıktısında bu sütunların `str` (metin) olduğunu görmüştük
— CSV formatı, Python listelerini kaydederken onları düz metne çevirir
(`"[52.22, 52.23, ...]"` gibi), okurken pandas bunu otomatik olarak
geri bir sayı dizisine çevirmez. Aşağıdaki birkaç hücre bunu doğruluyor:
`speed` hücresi boş bir liste string'i (`'[]'`), `latitude` hücresi ise
uzun bir metin (`len()` ile ölçülen 5576, gerçek nokta sayısı değil,
karakter sayısı).

In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 70591 entries, 0 to 70590
Data columns (total 10 columns):
 #   Column      Non-Null Count  Dtype
---  ------      --------------  -----
 0   longitude   70591 non-null  str  
 1   altitude    70591 non-null  str  
 2   latitude    70591 non-null  str  
 3   sport       70591 non-null  str  
 4   id          70591 non-null  int64
 5   heart_rate  70591 non-null  str  
 6   gender      70591 non-null  str  
 7   userId      70591 non-null  int64
 8   timestamp   70591 non-null  str  
 9   speed       70591 non-null  str  
dtypes: int64(2), str(8)
memory usage: 5.4 MB


In [10]:
df['speed'][0]

'[]'

In [11]:
type(df['speed'][0])

str

In [36]:
len(df['latitude'][0])

5576

## Metni sayı dizisine çevirme

`"[52.22, 52.23, ...]"` gibi bir string'i gerçek bir sayı dizisine
(numpy array) çevirmemiz lazım — yoksa üzerinde matematik yapamayız.
Yöntem basit: köşeli parantezleri (`[`, `]`) sil, virgülleri boşlukla
değiştir, kalan boşlukla ayrılmış sayıları `np.fromstring` ile tek
seferde oku. `arr[0:10]` çıktısı, bu dönüşümün gerçekten çalıştığını
ve gerçek enlem değerlerini (52.22... gibi) verdiğini doğruluyor.

In [40]:
cleaned = df['latitude'][0].replace("[", "").replace("]", "").replace(",", " ")
arr = np.fromstring(cleaned, sep=" ", dtype=np.float64)

In [47]:
arr[0:10]

array([52.2226809, 52.222727 , 52.2228258, 52.2228606, 52.2229289,
       52.2230171, 52.2231088, 52.2232156, 52.2232828, 52.2233734])

In [5]:
import ast 

## Bu dönüşümü tüm ilgili sütunlara uygula

Yukarıdaki dönüşümü tek satırlık bir fonksiyona (`parse_arr`) topluyoruz
ki `longitude`, `altitude`, `latitude`, `heart_rate`, `timestamp`,
`speed` sütunlarının HEPSİNE aynı işlemi tekrar tekrar yazmadan
uygulayabilelim (`num_cols` listesi + `.apply(parse_arr)`). İşlem
bittikten sonra `df.info()` ile bu sütunların artık `object` (yani
gerçek Python nesnesi — burada numpy array) tipinde olduğunu
doğruluyoruz; `sport` ve `gender` gibi sayısal olmayan sütunlar zaten
metin olarak kalmalı, onlara dokunmuyoruz.

In [6]:
def parse_arr(s):
    if isinstance(s, np.ndarray):
        return s.astype(np.float64)
    cleaned = s.replace("[", "").replace("]", "").replace(",", " ")
    return np.fromstring(cleaned, sep=" ", dtype=np.float64)


In [52]:
df.columns

Index(['longitude', 'altitude', 'latitude', 'sport', 'id', 'heart_rate',
       'gender', 'userId', 'timestamp', 'speed'],
      dtype='str')

In [7]:
num_cols = ['longitude', 'altitude', 'latitude','heart_rate', 'timestamp', 'speed']

In [8]:
for col in num_cols:
    df[col] = df[col].apply(parse_arr)

In [9]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 70591 entries, 0 to 70590
Data columns (total 10 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   longitude   70591 non-null  object
 1   altitude    70591 non-null  object
 2   latitude    70591 non-null  object
 3   sport       70591 non-null  str   
 4   id          70591 non-null  int64 
 5   heart_rate  70591 non-null  object
 6   gender      70591 non-null  str   
 7   userId      70591 non-null  int64 
 8   timestamp   70591 non-null  object
 9   speed       70591 non-null  object
dtypes: int64(2), object(6), str(2)
memory usage: 5.4+ MB


In [62]:
df[num_cols].head()

,longitude,altitude,latitude,heart_rate,timestamp,speed
0,"[6.8854929, 6.8853678, 6.8851621, 6.8848205, 6...","[-173.8, -151.2, -161.6, -165.4, -168.6, -172....","[52.2226809, 52.222727, 52.2228258, 52.2228606...","[80.0, 81.0, 94.0, 100.0, 102.0, 112.0, 108.0,...","[1397079200.0, 1397079210.0, 1397079220.0, 139...",[]
1,"[6.9144073, 6.9142929, 6.9141539, 6.9140268, 6...","[57.8, 57.6, 57.0, 56.4, 55.8, 55.2, 54.4, 53....","[52.2111711, 52.2112631, 52.2114064, 52.211608...","[60.0, 62.0, 92.0, 92.0, 132.0, 150.0, 150.0, ...","[1393908530.0, 1393908540.0, 1393908550.0, 139...",[]
2,"[6.9141348, 6.9145702, 6.9151684, 6.9158377, 6...","[22.8, 26.4, 30.8, 35.6, 43.0, 48.4, 49.8, 49....","[52.2110297, 52.2106325, 52.2102453, 52.209833...","[77.0, 93.0, 107.0, 121.0, 118.0, 120.0, 120.0...","[1393687930.0, 1393687950.0, 1393687970.0, 139...",[]
3,"[6.8678543, 6.8678634, 6.8675429, 6.8672183, 6...","[35.4, 35.2, 34.6, 34.2, 35.0, 35.2, 34.8, 34....","[52.1936673, 52.1934354, 52.1931993, 52.192873...","[75.0, 101.0, 116.0, 120.0, 124.0, 126.0, 127....","[1392480160.0, 1392480180.0, 1392480190.0, 139...",[]
4,"[6.9143328, 6.9146396, 6.9148949, 6.9151568, 6...","[63.0, 65.2, 66.0, 66.2, 65.8, 65.8, 67.0, 67....","[52.2112195, 52.2110264, 52.2108135, 52.210601...","[58.0, 83.0, 112.0, 115.0, 117.0, 116.0, 141.0...","[1392180430.0, 1392180440.0, 1392180450.0, 139...",[]


## Ara kayıt

Parse işlemi (string → array) tüm veri seti için biraz zaman alıyor;
bunu her oturumda tekrar tekrar yapmamak için sonucu aynı CSV dosyasının
üzerine kaydediyoruz. Not: CSV formatı array'leri yine metin olarak
saklar (numpy array'i CSV'ye direkt yazmanın bir yolu yok) — yani
dosyayı tekrar okuduğumuzda `parse_arr` adımını yine çalıştırmamız
gerekecek, ama en azından ham JSON'dan buraya kadar olan tüm önceki
adımları (indirme, ayrıştırma, filtreleme) tekrarlamamış oluyoruz.

In [10]:
df.to_csv(PATH)

## Haversine formülü — iki GPS noktası arası mesafe nasıl hesaplanır

### Neden basit bir "düz çizgi mesafesi" (Öklid mesafesi) kullanamayız

Enlem (latitude) ve boylam (longitude), Dünya yüzeyindeki bir noktayı
derece cinsinden tarif eder. İki nokta arasındaki mesafeyi bulmak için
"enlem farkının karesi + boylam farkının karesi, karekökü" gibi düz bir
(Öklid) formül kullanmak isteyebilirsiniz — ama bu YANLIŞ sonuç verir,
çünkü:

1. **Dünya düz değil, küre** — enlem/boylam dereceleri düz bir
   koordinat sistemi değil, kürenin yüzeyindeki açısal konumlardır.
   Kürenin yüzeyinde "en kısa yol" bir doğru değil, bir yay (büyük
   çember üzerindeki eğri) izler.
2. **1 derece boylam, her yerde aynı mesafeye karşılık gelmez** —
   Ekvator'da 1° boylam ~111 km'ye karşılık gelirken, kutuplara
   yaklaştıkça meridyenler birbirine yaklaşır ve aynı 1° boylam çok
   daha az fiziksel mesafeye karşılık gelir (kutup noktasında sıfıra
   iner). 1° enlem ise her yerde yaklaşık aynı mesafededir (~111 km).
   Yani enlem ve boylam eksenleri "eşit ağırlıklı" değildir — boylamın
   gerçek mesafe karşılığı, bulunduğunuz enleme bağlıdır.

Haversine formülü tam olarak bunu hesaba katan, küre yüzeyi için
matematiksel olarak doğru bir mesafe formülüdür.

### Formülün adım adım anlamı

```python
def haversine_np(lat1, lon1, lat2, lon2):
    R = 6371000.0  # Dünya yarıçapı (metre)
    phi1, phi2 = np.radians(lat1), np.radians(lat2)
    dphi = np.radians(lat2 - lat1)
    dlambda = np.radians(lon2 - lon1)

    a = np.sin(dphi / 2.0)**2 + np.cos(phi1) * np.cos(phi2) * np.sin(dlambda / 2.0)**2
    return 2 * R * np.arctan2(np.sqrt(a), np.sqrt(1 - a))
```

- **`R = 6371000.0`** — Dünya'nın ortalama yarıçapı, metre cinsinden.
  Küre üzerindeki bir yay uzunluğunu bulmak için gereken temel ölçek.
- **`np.radians(...)`** — trigonometrik fonksiyonlar (`sin`, `cos`)
  matematikte radyan cinsinden açı bekler, ama GPS koordinatları derece
  cinsinden verilir (örn. enlem 52.22°). Bu satır dereceyi radyana
  çevirir (derece × π/180).
- **`phi1, phi2`** — iki noktanın enlemleri (radyan cinsinden); coğrafi
  literatürde enlem geleneksel olarak yunanca φ (phi) harfiyle
  gösterilir.
- **`dphi`, `dlambda`** — iki nokta arasındaki enlem ve boylam farkı
  (radyan cinsinden); boylam farkı için λ (lambda) harfi kullanılır.
- **`a = sin²(Δφ/2) + cos(φ1)·cos(φ2)·sin²(Δλ/2)`** — bu, "haversine"
  (yarım-versine) fonksiyonunun kendisi. `cos(φ1)·cos(φ2)` çarpanı, tam
  olarak yukarıda bahsettiğimiz "boylamın gerçek mesafe karşılığı
  enleme bağlıdır" gerçeğini formüle sokan kısım — Ekvator'a yakın
  (`cos` büyük) boylam farkları daha çok ağırlık alır, kutuplara yakın
  (`cos` küçük) boylam farkları daha az ağırlık alır.
- **`2 * R * arctan2(√a, √(1-a))`** — `a` değerini, küre üzerindeki
  gerçek yay uzunluğuna (metre) çevirir. `arctan2` kullanılması (düz
  `arcsin` yerine), sayısal olarak daha kararlı bir sonuç verir —
  özellikle çok yakın ya da çok uzak noktalarda.

**Kısacası:** bu formül, iki enlem/boylam çiftini alıp, "bir kuşun
düz uçarak kat edeceği mesafe" değil, "Dünya küresinin yüzeyindeki en
kısa yol mesafesini" metre cinsinden verir — GPS tabanlı hız
hesaplamaları için gereken de tam olarak budur.

In [4]:
def haversine_np(lat1, lon1, lat2, lon2):
    """Ardışık GPS koordinatları arasındaki mesafeyi (metre) hesaplar."""
    R = 6371000.0  # Dünya yarıçapı (metre)
    phi1, phi2 = np.radians(lat1), np.radians(lat2)
    dphi = np.radians(lat2 - lat1)
    dlambda = np.radians(lon2 - lon1)

    a = np.sin(dphi / 2.0)**2 + np.cos(phi1) * np.cos(phi2) * np.sin(dlambda / 2.0)**2
    return 2 * R * np.arctan2(np.sqrt(a), np.sqrt(1 - a))

## Haversine'i tek bir nokta çifti üzerinde dene

Karmaşık bir fonksiyonu doğrudan 70591 satıra uygulamak yerine, önce
TEK bir antrenmanın İLK İKİ GPS noktası üzerinde deniyoruz. Bu, hem
fonksiyonun doğru çalıştığını hızlıca doğrulamamızı sağlıyor, hem de
"sonuç makul mü?" sorusunu kolayca cevaplayabiliyoruz: aşağıdaki
hücrede çıkan `9.94` değeri METRE cinsinden — yani bu iki GPS noktası
arasında ~10 metrelik bir mesafe var, ki iki ardışık GPS örneği (birkaç
saniye arayla alınmış) arasında beklenen mertebe tam olarak bu.

In [63]:
lat1 = df['latitude'][0][0]
lat2 = df['latitude'][0][1]
lon1 = df['longitude'][0][0]
lon2 = df['longitude'][0][1]


In [67]:
speed = haversine_np(lat1,lon1,lat2,lon2)

In [68]:
speed

np.float64(9.94446347942417)

## GPS sinyalinde tek noktalık sıçrama var mı?

Gerçek dünyada GPS ölçümü zaman zaman hatalı bir konum verebilir
(bina/ağaç gölgesi, sinyal yansıması gibi sebeplerle) — bu, tek bir
noktanın gerçek konumdan uzağa "sıçramasına" yol açar. Böyle bir nokta,
hız hesaplamasında devasa ve gerçek dışı bir sıçrama üretir (aniden
saniyede yüzlerce metre gitmiş gibi görünür).

Bunu tespit etmenin mantığı basit ama etkili: eğer `i` noktası
gerçekten bozuksa, hem `i-1 → i` arası hız anormal yüksek çıkar, hem de
`i → i+1` arası hız anormal yüksek çıkar — AMA `i` noktasını tamamen
ATLAYIP `i-1 → i+1` arası hızı hesaplarsak, bu normal bir değere döner
(çünkü aradaki gerçek konum değişmemiş, sadece `i` noktası yanlış
kaydedilmiş). Yani "önce hızlı, sonra hızlı, ama atlayınca normal" —
işte tam da bu üçünü karşılaştırarak `i` noktasının bozuk olup
olmadığını anlıyoruz. `MAX_MPS` (38 km/h'nin m/s karşılığı — elit
sprint tavanı) burada "anormal hızlı" için fiziksel bir sınır olarak
kullanılıyor.

In [12]:
row = df.iloc[0]
lat, lon, ts = row["latitude"], row["longitude"], row["timestamp"]

MAX_SPEED_KMH = 38.0                 # elit sprint tavanı (~10.5 m/s)
MAX_MPS = MAX_SPEED_KMH / 3.6

d_direct = haversine_np(lat[:-1], lon[:-1], lat[1:], lon[1:])   # metre, i -> i+1
dt_direct = np.diff(ts)
dt_direct = np.where(dt_direct <= 0, 1e6, dt_direct)             # dt=0 ayrı sorun, "hızlı" sayma

d_skip = haversine_np(lat[:-2], lon[:-2], lat[2:], lon[2:])     # metre, i -> i+2 (i'yi atlayarak)
dt_skip = ts[2:] - ts[:-2]
dt_skip = np.where(dt_skip <= 0, 1e6, dt_skip)

fast_in = (d_direct[:-1] / dt_direct[:-1]) > MAX_MPS    # i-1 -> i hızlı mı
fast_out = (d_direct[1:] / dt_direct[1:]) > MAX_MPS     # i -> i+1 hızlı mı
skip_ok = (d_skip / dt_skip) <= MAX_MPS                  # i'yi atlayınca (i-1 -> i+1) normal mi

spike = np.zeros(len(lat), dtype=bool)
spike[1:-1] = fast_in & fast_out & skip_ok   # hem önce hem sonra hızlı, ama atlayınca normal -> tek nokta bozuk

print("şüpheli GPS noktası sayısı:", spike.sum(), "/", len(lat))
print("indeksler:", np.where(spike)[0][:15])

şüpheli GPS noktası sayısı: 0 / 500
indeksler: []


## Her şeyi tek bir fonksiyonda topla

Şu ana kadar parça parça denediğimiz adımları (mesafe hesabı → ham hız
→ veri kalitesi düzeltmesi → yumuşatma) tek bir fonksiyonda
(`compute_speed_kmh`) birleştiriyoruz. Fonksiyonun içindeki her adımı
aşağıda açıklıyoruz.

### 1) Ardışık mesafe ve zaman farkı

```python
dist = haversine_np(lat[:-1], lon[:-1], lat[1:], lon[1:])
dt = np.diff(ts)
```

`lat[:-1]` dizinin ilk N-1 elemanı, `lat[1:]` ise 2.'den N'ye kadar
olan elemanları — ikisini haversine'e birlikte verince, `dist[i]` =
nokta `i` ile nokta `i+1` arasındaki mesafe (metre) olur. `np.diff(ts)`
aynı mantıkla ardışık zaman farklarını (saniye) verir. İkisi de
uzunluk N-1 (N nokta arasında N-1 "aralık" vardır).

### 2) Ham hız = mesafe / zaman — ama sıfıra bölme tuzağı var

```python
valid = dt > 0
speed = np.where(valid, (dist / np.where(valid, dt, 1)) * 3.6, np.nan)
```

Hız = mesafe/zaman, sonra `× 3.6` ile m/s'den km/h'ye çeviriyoruz. Ama
GPS verisinde **aynı saniyeye ait birden fazla ölçüm** (dt=0, tekrarlanan
zaman damgası) sıkça görülüyor — bu veri setinde ~%22 oranında.
Sıfıra bölmek matematiksel olarak tanımsızdır (`inf`/hata verir). Bu
satır iki katmanlı bir koruma uyguluyor: `np.where(valid, dt, 1)`
bölme işlemi SIRASINDA `dt=0` olan yerlere geçici olarak `1` koyuyor
(sadece hesaplama çökmesin diye — bu değeri zaten bir sonraki adımda
atacağız); dıştaki `np.where` ise gerçek sonuçta `dt<=0` olan yerlere
`NaN` ("tanımsız") yazıyor.

### 3) Tanımsız noktaları komşulardan tahmin et (interpolasyon)

```python
idx = np.arange(len(speed))
nan_mask = np.isnan(speed)
if nan_mask.any() and (~nan_mask).any():
    speed[nan_mask] = np.interp(idx[nan_mask], idx[~nan_mask], speed[~nan_mask])
```

`dt=0` olan noktalarda hız gerçekten tanımsız — oraya rastgele/yapay
büyük bir sayı koymak (ilk denemelerimizde `dt=0` yerine `0.001`
kullanmıştık) sahte, devasa hız sıçramalarına yol açtı (155.000
km/h gibi absürt değerler gördük). Doğru çözüm: bu noktaları boş
bırakıp (`NaN`), soldan ve sağdan en yakın GEÇERLİ (NaN olmayan) hız
değerleri arasında **doğrusal interpolasyon** yapmak — yani "bu iki
gerçek ölçüm arasında mantıklı bir ara değer" tahmin etmek. `np.interp`
tam bunu yapıyor.

### 4) Uzunluğu diğer sütunlarla hizala

```python
speed = np.insert(speed, 0, speed[0])
```

`speed` dizisi hep N-1 uzunlukta kalır (çünkü ardışık ÇİFTLER arasında
hesaplanıyor). Ama bize, her GPS noktası için bir hız değeri (N uzunluk)
lazım — DataFrame'in diğer sütunlarıyla (lat, hr, vb.) satır satır
hizalı olsun diye. En basit çözüm: ilk noktanın hızını, başa bir kez
daha ekleyerek diziyi N'ye tamamlamak.

### 5) Fiziksel olmayan uç değerleri kırp

```python
speed = np.clip(speed, 0.0, max_speed_kmh)
```

`max_speed_kmh=38.0` km/h — elit bir sprinter'ın bile aşamayacağı bir
tavan (~10.5 m/s). Tek bir kötü GPS okuması hâlâ anlık aşırı bir hız
üretebilir; bu satır böyle uç değerleri makul bir üst sınırda keserken,
negatif hızı da (fiziksel olarak anlamsız) alt sınırda (0) tutuyor.

### 6) Savitzky-Golay filtresi ile yumuşatma

```python
if len(speed) >= window_length and window_length > polyorder:
    speed = savgol_filter(speed, window_length, polyorder)
    speed = np.clip(speed, 0.0, max_speed_kmh)
```

Buraya kadarki adımlardan sonra bile hız verisi hâlâ "pürüzlü" —
GPS'in kendi ölçüm hassasiyeti sınırlı olduğu için, gerçekte sabit bir
tempoda koşan biri için bile hız değeri örnekten örneğe küçük küçük
zıplar. Savitzky-Golay filtresi, bu pürüzü bastırmak için kullanılan
özel bir yumuşatma yöntemi.

**Nasıl çalışır:** her hedef noktanın etrafında `window_length` (burada
7) kadar komşu örnek alır (3 önce, kendisi, 3 sonra), bu 7 noktaya
`polyorder` (burada 2, yani parabol) dereceden bir polinom eğrisi
uydurur (en küçük kareler yöntemiyle), sonra o eğrinin PENCERENİN TAM
ORTASINDAKİ değerini okur. Pencere bir sonraki noktaya kayar, aynı
işlem tekrarlanır.

**Neden düz "hareketli ortalama" (moving average) değil de bu:** düz
ortalama, pencere içindeki tüm noktaları eşit ağırlıkla toplayıp böler
— bu, gerçek hızlanma/yavaşlama eğilimini de düzleştirip yok eder
(örneğin bir sprint başlangıcının keskin yükselişini küntleştirir).
Savitzky-Golay ise komşu noktalara bir EĞRİ uydurduğu için, hem
gürültüyü bastırır hem de eğimi/ivmeyi (yani gerçek hızlanma
eğilimini) byük ölçüde korur — bu yüzden zaman serisi
yumuşatmasında hareketli ortalamaya göre daha sık tercih edilir.

**İki parametrenin kuralları:**
- `window_length` **tek sayı olmalı** — "pencerenin tam ortası" net bir
  nokta olsun diye (3 önce + kendisi + 3 sonra = simetrik 7; çift
  sayıda ortada iki nokta olur, belirsizlik çıkar).
- `window_length`, `polyorder`'dan **büyük olmalı** — bir polinom
  eğrisini benzersiz şekilde uydurmak için en az `polyorder+1` nokta
  gerekir (2. dereceden bir parabol için en az 3 nokta).

Fonksiyondaki `if` koşulu, bu iki kuralı ihlal eden durumlarda (çok
kısa bir antrenman, ya da hatalı parametre kombinasyonu) filtreyi
sessizce atlayıp hesaplamanın çökmesini önlüyor. Savgol sonrası tekrar
`clip` uygulanması ise güvenlik payı — dt=0 sorununu kökten çözdüğümüz
için artık ciddi bir taşma (overshoot) beklemiyoruz, ama garantisi yok.

In [ ]:
def compute_speed_kmh(lat, lon, ts, window_length=7, polyorder=2, max_speed_kmh=38.0):
    """
    GPS + zamandan hız türetir (km/h), dt=0 noktalarını interpole eder,
    uç değerleri kırpar ve Savitzky-Golay ile yumuşatır.
    """

    # delta phi (latitude farki) lat[:-1], lat[1:] ve delta lambda (longitude farki) lon[:-1], lon[1:]
    """lat[:-1] ile lat[1:]: biri baştan N-1, diğeri 2.'den N'ye kadar.
       Aralarındaki fark = ardışık nokta çiftleri. dist[i] = nokta i ile i+1 arası mesafe (metre).
       dt[i] aynı çift için zaman farkı (saniye). İkisi de uzunluk N-1."""
    
    dist = haversine_np(lat[:-1], lon[:-1], lat[1:], lon[1:])
    dt = np.diff(ts)

    """ Deneylerimizde satırların ~%22'sinde dt=0 (tekrarlanan timestamp) bulmuştuk. 
        dt=0'a bölmek inf/hata verir. np.where(valid, dt, 1) bölme sırasında dt=0 olan yerlere geçici 1 koyar (sadece hata çökmesin diye — sonucu zaten kullanmayacağız);
        dıştaki np.where ise gerçek sonuçta dt<=0 olan yerlere NaN yazar ("burada hız tanımsız"). *3.6: m/s → km/h. """
    valid = dt > 0
    speed = np.where(valid, (dist / np.where(valid, dt, 1)) * 3.6, np.nan)


    """ NaN'ları komşulardan interpole et"""
    idx = np.arange(len(speed))
    nan_mask = np.isnan(speed)
    if nan_mask.any() and (~nan_mask).any():
        speed[nan_mask] = np.interp(idx[nan_mask], idx[~nan_mask], speed[~nan_mask])

    """ speed hep N-1 uzunlukta (ardışık ÇİFTLER arası hesaplandığı için). 
        İlk noktanın hızını başa bir daha ekleyerek N'ye tamamlıyoruz — 
        DataFrame'in diğer sütunlarıyla (lat, hr...) hizalı olsun diye. """
    
    speed = np.insert(speed, 0, speed[0])         

    """ max_speed_kmh=38.0: elit sprint tavanı. 
        Tek bir kötü GPS okuması anlık devasa hız üretebilir; alt sınır da negatif hızı engelliyor. """
    
    speed = np.clip(speed, 0.0, max_speed_kmh)      

    """ polyorder genel kabuller 2-3, uygulanan parabolun derecesini belirler.
        window_length=7 noktalık kayan pencerede polyorder=2 (parabol) uydurup pencerenin ortasındaki değeri o eğriden okuyor 
        — düz hareketli ortalamadan farkı, ivmelenmeyi/yavaşlamayı koruması. 
        if: pencere veri boyundan büyükse veya derece pencereden büyükse (matematiksel olarak imkansız) filtreyi atla. 
        Savgol sonrası tekrar clip: dt=0 sorununu kökten çözdüğümüz için artık ciddi bir overshoot beklemiyoruz, ama güvenlik payı olarak kalıyor.
        her nokta için kayan bir pencere açar: window_length=7 demek, 
        her hedef noktanın etrafındaki 7 komşu örneği (3 önce, kendisi, 3 sonra) alıp bunlara polyorder dereceden bir polinom (en küçük kareler ile) uydurur, 
        sonra o polinomun pencerenin tam ortasındaki değerini okur. Sonra pencere bir sağa kayar, aynı işlem tekrarlanır.

        İki katı sınır var:

        Tek sayı olmalı — çünkü "pencerenin ortası" net bir nokta olmalı (3 önce + kendisi + 3 sonra = simetrik). Çift sayıda ortada iki nokta olur, belirsizlik çıkar.
        polyorder'dan büyük olmalı — bir polinomu benzersiz uydurmak için en az polyorder+1 nokta lazım (2. derece parabol için en az 3 nokta).
  """

    if len(speed) >= window_length and window_length > polyorder:
        speed = savgol_filter(speed, window_length, polyorder)
        speed = np.clip(speed, 0.0, max_speed_kmh)

    return speed




# tek satırda doğrula -- cell-21'deki sonuçla (min/max, uzunluk) tutarlı olmalı
test = compute_speed_kmh(lat, lon, ts)
print("uzunluk:", len(test), "min/max:", test.min(), test.max())
print("ilk 5:", np.round(test[:5], 2))

uzunluk: 500 min/max: 0.0 15.426963547407151
ilk 5: [2.9  4.97 6.41 7.23 7.46]


## Kritik Not:
Burada gözden kaçırılması kolay ama önemli bir şey var: window_length=7, örnek (index) cinsinden sabit — ama örnekler gerçek zamanda sabit aralıklı değil. Veri seti her antrenmanı, süresi ne olursa olsun, tam 500 noktaya yeniden örnekliyor (run_clustering_aerobic_anaerobic.py'nin kendi notunda da geçiyor: "2 saatlik koşuda noktalar ~14 sn arayla, 30 dakikalıkta ~3.6 sn arayla").

Bunun sonucu:

2 saatlik bir koşuda: 7 örnek × ~14 sn ≈ 98 saniye'lik bir zaman penceresinde yumuşatma yapıyorsunuz.
30 dakikalık bir koşuda: 7 örnek × ~3.6 sn ≈ 25 saniye'lik bir pencerede.
Yani aynı window_length=7, antrenmanın süresine göre gerçek dünyada çok farklı genişlikte bir yumuşatma uyguluyor — uzun koşularda ~4 kat daha geniş bir zaman aralığını harmanlıyorsunuz. Kısa, yoğun bir interval antrenmanında bu 98 saniyelik eşdeğer pencere (uzun koşularda) o kısa antrenmanda olsaydı muhtemelen tüm interval yapısını düzleştirip yok ederdi; ama süre farklı olduğu için farklı davranıyor. Bu, "sabit window_length her antrenmana adil davranıyor" varsayımının aslında yanlış olduğu, ileride (özellikle interval/tempo ayrımı gibi ince analizlerde) dikkat edilmesi gereken bir nokta — şu anki basit "eksik speed'i doldur" hedefi için sorun yaratmıyor, ama bilerek kullanmak lazım.

## Görsel/sayısal olarak son bir kontrol

Fonksiyonun ürettiği hız değerlerine bakarak bir mantık kontrolü daha
yapıyoruz: değerler makul bir aralıkta mı (aşırı sıçrama yok), ve
ardışık değerler arasında ani, anlamsız zıplamalar var mı? İlk 50
değeri yazdırıp göz kararı inceliyoruz — bu, ilerideki "tüm veri
setine uygula" adımından önceki son güven kontrolü.

In [14]:
print("ilk 50:", np.round(test[:50], 2))

ilk 50: [ 2.9   4.97  6.41  7.23  7.46  6.92  6.41  6.75  5.84  5.14  4.96  5.62
  7.1   8.41  8.13  8.21  8.33  8.49  8.88  8.95  8.93  8.91  8.73  8.22
  7.44  6.43  5.63  5.02  5.17  5.75  6.28  6.87  6.6   5.7   5.16  5.28
  5.93  7.11  7.41  7.26  6.85  6.49  6.92  7.79  8.92 10.27 11.48 11.28
  8.74  6.9 ]


## Tek satırdan tüm veri setine

`compute_speed_kmh`'i tek bir antrenman üzerinde geliştirip
doğruladıktan sonra, artık tüm 70591 satıra güvenle uygulayabiliriz.
Her satırın `latitude`, `longitude`, `timestamp` dizilerini fonksiyona
veriyoruz, dönen hız dizisini `speed` sütununa yazıyoruz — bu,
başlangıçta boş (`[]`) olan ~60000 satırın da artık dolu olduğu
anlamına geliyor. `len(df['speed'].iloc[0])` ve `.apply(len).gt(0).all()`
kontrolleri, hiçbir satırın boş kalmadığını ve tüm dizilerin beklenen
uzunlukta (500) olduğunu doğruluyor.

In [16]:
df["speed"] = [
    compute_speed_kmh(la, lo, t)
    for la, lo, t in zip(df["latitude"], df["longitude"], df["timestamp"])
]

print("örnek uzunluk:", len(df['speed'].iloc[0]))
print("tüm satırlar dolu mu:", df['speed'].apply(len).gt(0).all())

örnek uzunluk: 500
tüm satırlar dolu mu: True


In [17]:
df['speed']

0        [2.904172659167949, 4.967736122355964, 6.40813...
1        [4.195582786949944, 5.490696289211518, 6.64752...
2        [7.792656473403963, 10.881249650528808, 13.008...
3        [3.9874628995724044, 7.462747794301583, 10.095...
4        [10.820193769167016, 10.74224922091441, 10.668...
                               ...                        
70586    [12.39066754902334, 11.850809431323903, 11.470...
70587    [8.284418662704999, 8.093938706428782, 8.11820...
70588    [3.493449181258535, 3.5499078729660005, 3.5710...
70589    [1.0840414237528022, 1.3248870177240877, 1.613...
70590    [2.9046363481270574, 3.3392726412700022, 3.722...
Name: speed, Length: 70591, dtype: object

## Sonucu kaydet

Artık tüm antrenmanların hız verisi dolu; bunu ayrı bir dosyaya
(`endomondoHR_speed.csv`) kaydediyoruz — orijinal `endomondoHR_proper.csv`
dosyasının üzerine YAZMIYORUZ, böylece hız verisi olmadan önceki hâl
de korunmuş oluyor. Bu yeni dosya, projenin bir sonraki adımında
(`5_polarize.ipynb` — aerobik/anaerobik sınıflandırma) ana veri
kaynağı olarak kullanılacak.

In [18]:
df.to_csv('/home/can/zero-to-ai-architect/runsight/endomondoHR_speed.csv')